# 03 · Environmental distributions and principal component analysis

Separate water and soil PCAs describe environmental variation in `PCA.xlsx`.
Each sheet has **80 records = 2 lakes × 2 years × 4 seasons × 5 sites**.
All measured columns are included in their respective PCA; identifiers are excluded.
The soil PCA contains SOC concentration as supplied, not calculated soil carbon stock.

Run **Kernel → Restart Kernel and Run All Cells** using **Python (ler)**, with this notebook
in `extra_ipynb_files/`, and the three Excel files and `ecological_data.py` in the parent project folder. Each notebook runs independently.
Figures are displayed here and saved as 300-dpi PNGs and vector PDFs under `figures/` in the parent project folder.
The source workbooks are read only. Tables are saved under `tables/` in the parent project folder.

## Ecological context and questions

Loktak and Pumlen differ in hydrology, vegetation and human pressures, potentially producing different water and soil physicochemical signatures. These figures describe **seasonal distributions** and the dominant **multivariate environmental gradients**, rather than directly estimating the causal controls on carbon storage.

Each compartment (water, soil) has 80 observations representing two wetlands, two years, four seasons and five sites. In the PCA, environmental variables are standardised because they have different units. A PCA component is an orthogonal linear combination of standardised measurements, not a measured ecological process. PC1/PC2 separation is descriptive and requires independent testing if used to claim statistically distinct groups.

**Design limitation:** The site IDs recur between sampling periods, so visual separation should not be mistaken for independent replication. Environmental variables without verified measurement units are deliberately not assigned invented units.

## Choices

- Each variable gets its own distribution axis. Units absent from the source headers
  are not inferred. Box plots show quartiles and 1.5-IQR whiskers; overlaid dots show
  all observations, pooling the two years (10 records per lake–season, not 10 independent sites).
- Log display is available through `SCALE_OVERRIDES` in the setup cell. The default
  positive-range threshold is 20-fold, excluding pH and temperature. None of the
  current variables needs log scaling after separating the axes.
- PCA uses **untransformed measurements, centred and divided by sample SD**. No
  log transformation, imputation, outlier deletion or variable selection is performed.
- The code labels are cross-checked against the named blocks in `Scatter Plot.xlsx`.

In [ ]:
FIGURE_GROUP = "03_environment_pca"

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from IPython.display import display
# Find ecological_data.py in the project root, even when this notebook is in
# extra_ipynb_files/. Works with VS Code kernels started in either directory.
PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd() / "extra_ipynb_files"]
PROJECT_ROOT = next((p.resolve() for p in PROJECT_CANDIDATES
                     if (p / "ecological_data.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find ecological_data.py. Open VS Code from the project folder "
        "(the folder containing the Excel files and ecological_data.py)."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ecological_data import (
    ROOT, KEYS, SEASONS, WETLANDS, load_scatter, load_pca,
    decode_pca, load_line_summary, load_heatmaps,
)

FIG_DIR = ROOT / "figures" / FIGURE_GROUP
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = ROOT / "tables"
TABLE_DIR.mkdir(exist_ok=True)
sns.set_theme(style="ticks", context="paper", palette="colorblind", rc={
    "font.size": 10, "axes.labelsize": 11, "axes.titlesize": 12,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.fontsize": 9, "legend.title_fontsize": 10,
    "figure.dpi": 110, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42,
})
PALETTE = dict(zip(WETLANDS, sns.color_palette("colorblind", 2)))
MARKERS = dict(zip(SEASONS, ["o", "s", "^", "D"]))
LABELS = {
    "Soil_stock": r"Soil carbon stock (Mg C ha$^{-1}$)",
    "Phyto_carbon": "Phytoplankton carbon\n(unit not supplied)",
    "TBC": r"Total biomass carbon (Mg C ha$^{-1}$)",
    "Soil_Temp": "Soil temperature", "Water_Temp": "Water temperature",
    "Soil_pH": "Soil pH", "Water_pH": "Water pH", "BD": "Bulk density",
    "Moisture": "Soil moisture", "SOC": "Soil organic carbon (SOC)",
    "N": "Available nitrogen", "P": "Available phosphorus", "K": "Available potassium",
    "TDS": "Total dissolved solids (TDS)", "EC": "Electrical conductivity (EC)",
    "CO2": r"Free CO$_2$", "Turbidity": "Turbidity", "DO": "Dissolved oxygen (DO)",
    "BOD": "Biochemical oxygen demand", "COD": "Chemical oxygen demand",
    "Alkalinity": "Alkalinity", "Hardness": "Total hardness",
    "Water_K": "Water potassium", "Inorg_P": "Inorganic phosphorus",
}

def save_show(fig, name):
    """Save both a 300-dpi PNG and a vector PDF; retain the inline notebook plot."""
    for extension in ["png", "pdf"]:
        fig.savefig(FIG_DIR / f"{name}.{extension}", bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

# Display-only choices: no log transform is applied to correlation or PCA inputs.
# Set an entry to "log" or "linear" to override the default for that variable.
SCALE_OVERRIDES = {}
def axis_scale(values, variable):
    values = np.asarray(values, dtype=float)
    assert np.isfinite(values).all()
    if variable in SCALE_OVERRIDES:
        choice = SCALE_OVERRIDES[variable]
    elif "pH" in variable or "Temp" in variable:
        choice = "linear"
    else:
        choice = "log" if values.min() > 0 and values.max()/values.min() >= 20 else "linear"
    assert choice in {"linear", "log"}
    if choice == "log" and np.any(values <= 0):
        raise ValueError(f"Cannot use log scale for nonpositive {variable} values.")
    return choice

def set_numeric_axis(ax, axis, values, variable):
    scale = axis_scale(values, variable)
    getattr(ax, f"set_{axis}scale")(scale)
    getattr(ax, f"set_{axis}label")(LABELS.get(variable, variable) + (" [log scale]" if scale == "log" else ""))

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | seaborn {sns.__version__}")
print(f"Figures: {FIG_DIR}")

In [ ]:
raw = load_pca()
frames = {name:decode_pca(frame) for name,frame in raw.items()}
scatter, _ = load_scatter()
for name, frame in frames.items():
    shared = [c for c in frame if c in scatter and c not in KEYS]
    paired = frame.merge(scatter, on=KEYS, suffixes=("_pca","_named"),validate="one_to_one")
    for variable in shared:
        assert np.allclose(paired[variable+"_pca"],paired[variable+"_named"],rtol=0,atol=1e-10)

variables = {name:[c for c in frame if c not in KEYS] for name,frame in frames.items()}
scale_rows = []
for name,frame in frames.items():
    for variable in variables[name]:
        scale_rows.append({"Compartment":name,"Variable":variable,"Minimum":frame[variable].min(),
                           "Maximum":frame[variable].max(),"Display axis":axis_scale(frame[variable],variable)})
display(pd.DataFrame(scale_rows))
print("Missing values: 0; duplicate sampling keys: 0 in each sheet.")

## 1. Distributions by season and wetland

In [ ]:
def distribution_page(compartment, selected, page):
    frame = frames[compartment]
    columns = min(2, len(selected))
    rows = int(np.ceil(len(selected)/columns))
    fig, axes = plt.subplots(rows, columns, figsize=(7.3, 3.35*rows),
                             squeeze=False, layout="constrained")
    for ax, variable in zip(axes.flat, selected):
        sns.boxplot(data=frame, x="Season", y=variable, hue="Wetland", order=SEASONS,
                    hue_order=WETLANDS, palette=PALETTE, showfliers=False, linewidth=1.5,
                    width=0.7, saturation=0.65, legend=False, ax=ax)
        # Reproducible horizontal jitter; no change to the measured y values.
        np.random.seed(2718)
        sns.stripplot(data=frame, x="Season", y=variable, hue="Wetland", order=SEASONS,
                      hue_order=WETLANDS, palette=PALETTE, dodge=True, jitter=0.12,
                      size=5, alpha=0.75, linewidth=0.4, edgecolor="white", legend=False, ax=ax)
        set_numeric_axis(ax, "y", frame[variable], variable)
        ax.set_xlabel("")
        ax.set_xticks(range(4), ["Pre-\nmonsoon","Monsoon","Post-\nmonsoon","Winter"])
        ax.grid(axis="y", alpha=0.2)
    for ax in list(axes.flat)[len(selected):]: ax.set_visible(False)
    handles = [Line2D([],[],color=PALETTE[lake],marker="o",linestyle="",label=lake) for lake in WETLANDS]
    fig.legend(handles=handles, loc="outside lower center", ncol=2, frameon=False, markerscale=1.5)
    fig.suptitle(f"{compartment}: seasonal distributions (both years)", fontsize=12)
    save_show(fig, f"{compartment.lower()}_distributions_{page}")

for page,start in enumerate(range(0,len(variables["Soil"]),4),1):
    distribution_page("Soil", variables["Soil"][start:start+4], page)

In [ ]:
for page,start in enumerate(range(0,len(variables["Water"]),4),1):
    distribution_page("Water", variables["Water"][start:start+4], page)

## 2. PCA definition and scaling

For each compartment, with $n=80$ and $p$ measured variables,
\[
Z_{ij}=\frac{X_{ij}-\bar X_j}{s_j},\quad
s_j^2=\frac{1}{n-1}\sum_i(X_{ij}-\bar X_j)^2,\quad
Z=UDV^{\mathsf T}.
\]
Scores are $T=ZV=UD$, eigenvalues are $\lambda_k=D_{kk}^2/(n-1)$,
and explained fractions are $\lambda_k/\sum_\ell\lambda_\ell$.
Standardisation gives each variable unit sample variance, so $\sum_k\lambda_k=p$.

Here **weights** mean $V_{jk}$; **correlation loadings** mean
$L_{jk}=\mathrm{corr}(Z_j,T_k)=V_{jk}\sqrt{\lambda_k}$.
These are saved separately to avoid conflating conventions. Signs of an entire PC
can reverse without changing the PCA. Correlated inputs (e.g. EC and TDS) remain
included and can give their shared environmental pattern more representation.

In [ ]:
results = {}
for name, frame in frames.items():
    X = frame[variables[name]].astype(float)
    means, scales = X.mean(), X.std(ddof=1)
    assert (scales > 0).all()
    Z = (X-means)/scales
    U, singular_values, Vt = np.linalg.svd(Z.to_numpy(), full_matrices=False)
    scores = U * singular_values
    eigenvalues = singular_values**2 / (len(Z)-1)
    explained_fraction = eigenvalues / eigenvalues.sum()
    pcs = [f"PC{k+1}" for k in range(X.shape[1])]
    weights = pd.DataFrame(Vt.T, index=X.columns, columns=pcs)
    loadings = weights.mul(np.sqrt(eigenvalues), axis="columns")
    score_frame = pd.concat([frame[KEYS].reset_index(drop=True),pd.DataFrame(scores,columns=pcs)],axis=1)
    variance = pd.DataFrame({"PC":pcs, "Eigenvalue":eigenvalues,
                             "Explained (%)":100*explained_fraction,
                             "Cumulative (%)":100*np.cumsum(explained_fraction)})
    # Numerical checks: complete PCA reconstructs Z and the loadings are correlations.
    assert np.allclose(scores @ Vt,Z)
    assert np.isclose(eigenvalues.sum(),X.shape[1])
    direct = np.corrcoef(Z.to_numpy().T,scores.T)[:X.shape[1],X.shape[1]:]
    assert np.allclose(loadings,direct)
    results[name] = dict(scores=score_frame,weights=weights,loadings=loadings,variance=variance)
    for suffix,table in [("scores",score_frame),("weights",weights),("correlation_loadings",loadings),
                          ("variance",variance),("standardisation",pd.DataFrame({"Mean":means,"Sample_SD":scales}))]:
        table.to_csv(TABLE_DIR / f"pca_{name.lower()}_{suffix}.csv",index=suffix not in ["scores","variance"])
    print(f"{name}: PC1+PC2 = {variance['Explained (%)'].iloc[:2].sum():.2f}%")
    display(variance.round(3))

## 3. Scree and score plots

The scree plot shows how much variance each component retains. Points in the scores
plot are individual records, coloured by lake and shaped by season. Nearby points
have similar projections onto PC1–PC2; differences in the remaining PCs are not shown.
Both years are pooled here; the following figure separates them.

In [ ]:
def pca_overview(name):
    result=results[name]
    variance=result["variance"]
    pc1,pc2=variance["Explained (%)"].iloc[:2]
    fig,axes=plt.subplots(1,2,figsize=(8.0,4.4),layout="constrained",gridspec_kw={"width_ratios":[1,1.3]})
    sns.barplot(data=variance,x="PC",y="Explained (%)",color=PALETTE["Loktak"],ax=axes[0])
    
    axes[0].set(title=f"{name}: explained variance",xlabel="Principal component",ylabel="Variance (%)",ylim=(0,105))
    axes[0].tick_params(axis="x",rotation=45)
    
    sns.scatterplot(data=result["scores"],x="PC1",y="PC2",hue="Wetland",style="Season",
                    hue_order=WETLANDS,style_order=SEASONS,palette=PALETTE,markers=MARKERS,
                    s=100,alpha=0.85,edgecolor="white",ax=axes[1])
    axes[1].set(title=f"{name}: PCA scores",xlabel=f"PC1 ({pc1:.1f}%)",ylabel=f"PC2 ({pc2:.1f}%)")
    axes[1].axhline(0,color="0.65",lw=1); axes[1].axvline(0,color="0.65",lw=1)
    axes[1].set_aspect("equal",adjustable="datalim")
    sns.move_legend(axes[1],"upper left",bbox_to_anchor=(1.02,1),frameon=False)
    save_show(fig,f"pca_{name.lower()}_scree_scores")

pca_overview("Soil")
pca_overview("Water")

In [ ]:
def pca_by_year(name):
    result=results[name]
    pc1,pc2=result["variance"]["Explained (%)"].iloc[:2]
    fig,axes=plt.subplots(1,2,figsize=(7.1,4.8),sharex=True,sharey=True,layout="constrained")
    for ax,year in zip(axes,[2022,2023]):
        sns.scatterplot(data=result["scores"].query("Year == @year"),x="PC1",y="PC2",
                        hue="Wetland",style="Season",hue_order=WETLANDS,style_order=SEASONS,
                        palette=PALETTE,markers=MARKERS,s=100,legend=False,ax=ax)
        ax.axhline(0,color="0.65",lw=1); ax.axvline(0,color="0.65",lw=1)
        ax.set(title=str(year),xlabel=f"PC1 ({pc1:.1f}%)",ylabel=f"PC2 ({pc2:.1f}%)")
        ax.set_aspect("equal",adjustable="box")
    handles=[Line2D([],[],color=PALETTE[lake],marker="o",linestyle="",label=lake) for lake in WETLANDS]
    handles += [Line2D([],[],color="0.25",marker=MARKERS[s],linestyle="",label=s) for s in SEASONS]
    fig.legend(handles=handles,loc="outside lower center",ncol=3,frameon=False,markerscale=1.5)
    fig.suptitle(f"{name}: same PCA basis, separated by year",fontsize=12)
    save_show(fig,f"pca_{name.lower()}_scores_by_year")

pca_by_year("Soil")
pca_by_year("Water")

## 4. PCA loading directions and complete loading tables

For readability, the correlation circle labels the **five best-represented variables in PC1–PC2**,
ranked by $L_{j1}^2+L_{j2}^2$. This only selects displayed arrows; PCA still includes
every measured variable. Arrows use correlation loadings multiplied by a single
display factor, printed in the title. Their absolute lengths are not score units.

All variable–PC correlations are shown in the adjacent heatmap for the first four
PCs and saved for every PC. Similar arrow directions indicate similar projections;
interpretation is weaker when much of a variable's variance lies outside PC1–PC2.

In [ ]:
def pca_loading_figures(name):
    """Print-friendly PCA correlation circle and correlation-loading matrix.

    Scores and loading vectors are deliberately NOT overlaid: different
    numerical scales made arrows/labels illegible on A4-sized figures.
    """
    result = results[name]
    loadings = result["loadings"]
    pc1, pc2 = result["variance"]["Explained (%)"].iloc[:2]
    represented = loadings[["PC1", "PC2"]].pow(2).sum(axis=1)
    selected = represented.nlargest(5).index

    fig, ax = plt.subplots(figsize=(5.8, 5.8), layout="constrained")
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="0.65", lw=1))
    for i, variable in enumerate(selected, 1):
        x, y = loadings.loc[variable, ["PC1", "PC2"]]
        ax.annotate("", xy=(x, y), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color="0.2", lw=1.5))
        ax.text(x * 1.12, y * 1.12, str(i), ha="center", va="center",
                fontsize=10, bbox=dict(facecolor="white", edgecolor="0.8", pad=1.2))
    ax.axhline(0, color="0.8", lw=0.8); ax.axvline(0, color="0.8", lw=0.8)
    ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25),
           xlabel=f"Correlation with PC1 ({pc1:.1f}%)",
           ylabel=f"Correlation with PC2 ({pc2:.1f}%)",
           title=f"{name}: principal-component correlation circle")
    ax.set_aspect("equal")
    key = "\n".join(f"{i}. {v}" for i, v in enumerate(selected, 1))
    fig.text(0.5, 0.01, key, ha="center", va="bottom", fontsize=9)
    save_show(fig, f"pca_{name.lower()}_correlation_circle")

    # Comprehensive loadings in a separate figure, not compressed beside scores.
    fig, ax = plt.subplots(figsize=(6.7, max(4, 0.38*len(loadings))), layout="constrained")
    sns.heatmap(loadings.iloc[:, :4],
                cmap=sns.diverging_palette(250, 35, s=85, l=45, as_cmap=True),
                vmin=-1, vmax=1, center=0, annot=True, fmt=".2f",
                annot_kws={"fontsize":9}, linewidths=0.6,
                cbar_kws={"label":"Correlation with PC", "shrink":0.7}, ax=ax)
    ax.set(title=f"{name}: correlation loadings", xlabel="", ylabel="")
    ax.tick_params(axis="y", rotation=0)
    save_show(fig, f"pca_{name.lower()}_loading_heatmap")
    display(represented.rename("Variance fraction represented on PC1–PC2").to_frame().round(3))

pca_loading_figures("Soil")
pca_loading_figures("Water")

## Interpretation limits

PCA describes covariance of the included environmental variables. Lake or season
separation is descriptive and is not a significance test or evidence of environmental
control of carbon storage. Repeated sites remain repeated observations. The percentage
shown on each axis states how much total standardised variance that projection retains.
For carbon-pool relationships, return to the paired scatter plots in notebook 01.

## Conclusions and ecological interpretation

- Soil and water PCAs characterise **different environmental compartments**; compare their gradients conceptually rather than equating their component axes.
- The percentage printed on PC1 and PC2 gives the proportion of variance explained **within the standardised variables included in that PCA**. No carbon-stock variance explained or causal effect is implied.
- A point near another point has a similar multivariate environmental profile along the **displayed** components. Arrows represent correlations between variables and the principal components; their orientation is informative only to the extent that variables are represented well on PC1–PC2.
- The correlation circle labels only five variables to preserve print readability; consult the complete loading tables saved to CSV before asserting which variables dominate a component.
- Interpret any separation of wetlands or seasons as descriptive. Site-level repeated measures, temporal confounding, and correlated inputs (e.g., EC and TDS) should be addressed before formal ecological claims.